# Pneumonia Detection from Chest X-Rays using CNN + Grad-CAM

**College Project**

### What this notebook does
1. Loads the **PneumoniaMNIST** dataset (from the `medmnist` library) which contains chest X-ray images labeled as *Normal* or *Pneumonia*.
2. Trains a simple **Convolutional Neural Network (CNN)** to classify the images.
3. Evaluates the model with accuracy/loss curves, a classification report, and a confusion matrix.
4. Implements **Grad-CAM** (Gradient-weighted Class Activation Mapping) *from scratch* to visualize which parts of the X-ray the model is "looking at" when it makes a prediction.

### About the dataset
PneumoniaMNIST is part of the **MedMNIST** collection (https://medmnist.com/), a set of standardized biomedical image datasets. It is itself derived from a large public chest X-ray dataset. When we call the `medmnist` API below, it **automatically downloads the data from the official MedMNIST server (a direct download link) the first time it is run** — no manual download, Kaggle account, or API key is needed. The full dataset ships with proper train / validation / test splits (thousands of images total), so we are working with the complete dataset, not a toy subset.

| Split | # Images |
|-------|----------|
| Train | 4,708 |
| Validation | 524 |
| Test | 624 |

Each image is a grayscale chest X-ray, and the label is binary: `0 = Normal`, `1 = Pneumonia`.

## Step 0 — Install required libraries

Run this cell first. It installs everything we need: `tensorflow` (to build/train the CNN), `medmnist` (to download and load the dataset), `opencv-python-headless` (to resize images and blend the Grad-CAM heatmap onto the X-ray), `matplotlib` (for plots), `numpy` (array math), and `scikit-learn` (evaluation metrics).

We install using `sys.executable -m pip` (instead of plain `%pip` or `!pip`) so the packages are guaranteed to be installed into **this notebook's own kernel environment** — this avoids the common `ModuleNotFoundError` that happens when pip installs into a different Python than the one the kernel is running.

**If this is the first time you're installing these packages, restart the kernel after this cell finishes, then run the notebook again from the top** (Jupyter: *Kernel → Restart Kernel*, VS Code: the "Restart" button in the notebook toolbar). This ensures the kernel picks up the newly installed packages.

In [ ]:
# Install all required libraries into THIS kernel's Python environment.
# We use subprocess.run with an argument list (not a shell string) so that
# paths containing spaces (e.g. "C:\Program Files\...") don't break the command.
import sys
import subprocess

print("Kernel Python executable:", sys.executable)
print("Installing packages, this can take a few minutes (TensorFlow is large)...\n")

packages = ["tensorflow", "medmnist", "opencv-python-headless", "matplotlib", "numpy", "scikit-learn"]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + packages,
    capture_output=True, text=True
)

# Show pip's own output so any real error is visible instead of hidden
print(result.stdout[-4000:])
if result.returncode != 0:
    print("\n===== PIP INSTALL FAILED — stderr below =====")
    print(result.stderr[-4000:])
    print("===============================================")
    print("Copy the error above and fix it (e.g. no internet, disk space, permissions),")
    print("then re-run this cell. Do NOT continue to Step 1 until this succeeds.")
else:
    print("\nAll packages installed successfully.")
    print("If you are running this for the FIRST time, restart the kernel now")
    print("(Kernel -> Restart, or the Restart button in VS Code), then run the notebook from the top.")

## Step 1 — Imports

We import all the libraries we'll use throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import medmnist
from medmnist import PneumoniaMNIST

# Make results reproducible
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("MedMNIST version:", medmnist.__version__)

## Step 2 — Load the dataset

`PneumoniaMNIST(split=..., download=True)` downloads the dataset straight from the official MedMNIST download link (a `.npz` file hosted on Zenodo) the first time it runs, then caches it locally (`~/.medmnist/` — on Colab this is `/root/.medmnist`) so future runs are instant.

The MedMNIST library's built-in downloader sometimes hits a `504 Gateway Time-out` on Zenodo's servers (this is a flaky-server issue, not something wrong with your code or environment). To work around it, the next cell downloads the `.npz` file directly with **automatic retries and a long timeout**, and saves it straight into the same cache folder MedMNIST expects — so when we then construct `PneumoniaMNIST(...)`, it finds the file already there and just loads it, without touching the flaky downloader at all.

In [ ]:
# Load the three splits. Since we already downloaded the .npz file into
# MedMNIST's cache folder above, download=True here will find it locally
# and just load it — it won't hit the network/downloader again.
# as_rgb=False keeps images single-channel (grayscale), which is natural for X-rays.
train_dataset = PneumoniaMNIST(split="train", download=True, as_rgb=False)
val_dataset   = PneumoniaMNIST(split="val",   download=True, as_rgb=False)
test_dataset  = PneumoniaMNIST(split="test",  download=True, as_rgb=False)

# Pull the raw numpy arrays out of the dataset objects
X_train, y_train = train_dataset.imgs, train_dataset.labels
X_val,   y_val   = val_dataset.imgs,   val_dataset.labels
X_test,  y_test  = test_dataset.imgs,  test_dataset.labels

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
print("Label meaning:", train_dataset.info["label"])

In [ ]:
# Download (first run only) and load the three splits.
# as_rgb=False keeps images single-channel (grayscale), which is natural for X-rays.
train_dataset = PneumoniaMNIST(split="train", download=True, as_rgb=False)
val_dataset   = PneumoniaMNIST(split="val",   download=True, as_rgb=False)
test_dataset  = PneumoniaMNIST(split="test",  download=True, as_rgb=False)

# Pull the raw numpy arrays out of the dataset objects
X_train, y_train = train_dataset.imgs, train_dataset.labels
X_val,   y_val   = val_dataset.imgs,   val_dataset.labels
X_test,  y_test  = test_dataset.imgs,  test_dataset.labels

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
print("Label meaning:", train_dataset.info["label"])

## Step 3 — Visualize sample images

Before doing anything else, let's *look* at the data. We plot a grid of X-rays with their labels so we understand what the model will be learning from.

In [ ]:
class_names = ["Normal", "Pneumonia"]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap="gray")
    ax.set_title(class_names[int(y_train[i])])
    ax.axis("off")
plt.suptitle("Sample Chest X-Rays from PneumoniaMNIST")
plt.tight_layout()
plt.show()

# Check class balance
unique, counts = np.unique(y_train, return_counts=True)
print("Class distribution in training set:")
for u, c in zip(unique, counts):
    print(f"  {class_names[int(u)]}: {c} images")

## Step 4 — Preprocess the data

Two simple preprocessing steps:
1. **Normalize**: pixel values currently range 0–255. We scale them to 0–1, which helps the network train faster and more stably.
2. **Reshape**: Keras' `Conv2D` layers expect input shaped `(height, width, channels)`. The images already have a channel dimension, but we make sure the shape and data types are correct (`float32` for images, `int` for labels).

In [ ]:
# Normalize pixel values from [0, 255] to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_val   = X_val.astype("float32")   / 255.0
X_test  = X_test.astype("float32")  / 255.0

# Make sure images have shape (num_images, height, width, channels)
if X_train.ndim == 3:
    X_train = X_train[..., np.newaxis]
    X_val   = X_val[...,   np.newaxis]
    X_test  = X_test[...,  np.newaxis]

# Labels come as shape (N, 1); flatten them to shape (N,) and cast to int
y_train = y_train.flatten().astype("int32")
y_val   = y_val.flatten().astype("int32")
y_test  = y_test.flatten().astype("int32")

IMG_SHAPE = X_train.shape[1:]
print("Image shape used by the model:", IMG_SHAPE)
print("Pixel value range after normalization:", X_train.min(), "to", X_train.max())

## Step 5 — Build a simple CNN

We keep the architecture deliberately simple so it's easy to explain in class:

- **3 convolutional blocks**, each with: `Conv2D → ReLU → MaxPooling`. Each block learns increasingly complex patterns (edges → textures → shapes like lung opacities).
- The **last convolutional layer is named `last_conv`** — we need this specific layer later for Grad-CAM, since Grad-CAM looks at the gradients flowing into the final convolutional feature map.
- After the conv blocks, we flatten the feature maps and pass them through a couple of `Dense` layers to make the final Normal-vs-Pneumonia decision.
- The output layer has a single neuron with a **sigmoid** activation, since this is binary classification.

In [ ]:
def build_model(input_shape):
    # Built with the Functional API (instead of Sequential) so that every
    # layer has a proper graph connection from the start. This matters for
    # Grad-CAM later: we need to build a sub-model that reads the output of
    # `last_conv`, and Sequential models can raise
    # "AttributeError: layer has never been called" when you try that,
    # even after training. The Functional API avoids this issue entirely.
    inputs = keras.Input(shape=input_shape)

    # --- Conv block 1 ---
    x = layers.Conv2D(16, (3, 3), activation="relu", padding="same", name="conv1")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Conv block 2 ---
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Conv block 3 (this is the layer Grad-CAM will use) ---
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="last_conv")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Classifier head ---
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)  # binary output: probability of Pneumonia

    model = keras.Model(inputs, outputs)
    return model

model = build_model(IMG_SHAPE)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Step 6 — Train the model

We train for about **10 epochs**, using the validation set to monitor how well the model generalizes to unseen data during training.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

## Step 7 — Evaluate the model

We look at the model's performance from several angles:
1. **Accuracy / loss curves** — show whether the model is still improving, or over/under-fitting.
2. **Classification report** — precision, recall, and F1-score for each class (very important in medical settings, where missing a pneumonia case is worse than a false alarm).
3. **Confusion matrix** — a visual breakdown of correct vs. incorrect predictions.

In [ ]:
# --- Accuracy and loss curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Validation Loss")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Test set evaluation ---
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

# Get predictions (sigmoid gives a probability; threshold at 0.5 to get a class)
y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype("int32")

# --- Classification report ---
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix on Test Set")
plt.show()

## Step 8 — Grad-CAM from scratch

**What is Grad-CAM?** Grad-CAM (Gradient-weighted Class Activation Mapping) answers the question: *"Which regions of the input image influenced the model's decision the most?"* This is important in medical AI — we don't just want a prediction, we want to see whether the model is actually focusing on the lungs (and not, say, a corner artifact).

**How it works, step by step:**
1. Run the image through the network and record two things: the feature maps output by our last convolutional layer (`last_conv`), and the final prediction score.
2. Compute the **gradient** of the predicted class score with respect to those feature maps. This tells us how much each feature map "matters" for the prediction.
3. **Average each feature map's gradient** spatially to get one importance weight per channel.
4. Take a **weighted sum** of the feature maps using those importance weights — channels that matter more contribute more.
5. Apply **ReLU** (we only care about features that positively influence the predicted class) and normalize to 0–1 to get the final heatmap.

We implement this manually with `tf.GradientTape`, TensorFlow's tool for automatic differentiation.

In [ ]:
def compute_gradcam_heatmap(model, img_array, last_conv_layer_name="last_conv"):
    """
    Computes a Grad-CAM heatmap for a single image.

    Args:
        model: the trained Keras model.
        img_array: a single preprocessed image with shape (1, H, W, C).
        last_conv_layer_name: name of the last conv layer to inspect.

    Returns:
        A 2D numpy array (values in [0, 1]) representing the heatmap.
    """
    # Build a model that outputs both the last conv layer's feature maps
    # AND the final prediction, given the same input.
    grad_model = keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Use GradientTape to record operations so we can differentiate later
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array)
        # For binary sigmoid output, the "score" for the predicted class
        # is simply the single output neuron's value.
        class_score = predictions[:, 0]

    # Gradient of the class score w.r.t. the last conv layer's feature maps
    grads = tape.gradient(class_score, conv_output)

    # Average the gradients over width and height -> one weight per channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weighted sum of the feature maps (weighted by channel importance)
    conv_output = conv_output[0]  # remove the batch dimension
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Keep only positive influence (ReLU) and normalize to [0, 1]
    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.math.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val

    return heatmap.numpy()


def overlay_gradcam(original_img, heatmap, alpha=0.4):
    """
    Overlays a Grad-CAM heatmap on top of the original grayscale X-ray.

    Args:
        original_img: 2D array (H, W), pixel values in [0, 1].
        heatmap: 2D array from compute_gradcam_heatmap, values in [0, 1].
        alpha: blending strength of the heatmap.

    Returns:
        An RGB image (H, W, 3), uint8, ready to display with matplotlib.
    """
    h, w = original_img.shape[:2]

    # Resize the (small) heatmap up to the original image size
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)

    # Convert heatmap to a color image (red = high importance)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Convert original grayscale image to RGB so we can blend it with the heatmap
    original_uint8 = np.uint8(255 * original_img)
    original_rgb = cv2.cvtColor(original_uint8, cv2.COLOR_GRAY2RGB)

    # Blend the two images together
    overlay = cv2.addWeighted(original_rgb, 1 - alpha, heatmap_color, alpha, 0)

    return overlay

## Step 9 — Visualize Grad-CAM results on 5 test images

For 5 random test images, we show three things side by side:
- The original X-ray
- The raw Grad-CAM heatmap
- The heatmap overlaid on the X-ray

We also print the true label and the model's predicted label/probability for context.

In [ ]:
# Pick 5 random test images
np.random.seed(1)
sample_indices = np.random.choice(len(X_test), size=5, replace=False)

fig, axes = plt.subplots(5, 3, figsize=(9, 15))

for row, idx in enumerate(sample_indices):
    img = X_test[idx]                      # shape (H, W, 1), already normalized
    img_batch = np.expand_dims(img, axis=0)  # shape (1, H, W, 1) for the model

    # Model prediction
    prob = model.predict(img_batch, verbose=0)[0][0]
    pred_label = class_names[int(prob >= 0.5)]
    true_label = class_names[int(y_test[idx])]

    # Grad-CAM heatmap + overlay
    heatmap = compute_gradcam_heatmap(model, img_batch, last_conv_layer_name="last_conv")
    overlay = overlay_gradcam(img.squeeze(), heatmap)

    # Original image
    axes[row, 0].imshow(img.squeeze(), cmap="gray")
    axes[row, 0].set_title(f"Original\nTrue: {true_label}")
    axes[row, 0].axis("off")

    # Raw heatmap
    axes[row, 1].imshow(heatmap, cmap="jet")
    axes[row, 1].set_title("Grad-CAM Heatmap")
    axes[row, 1].axis("off")

    # Overlay
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"Overlay\nPred: {pred_label} ({prob:.2f})")
    axes[row, 2].axis("off")

plt.suptitle("Grad-CAM Visualizations on Test X-Rays", fontsize=14)
plt.tight_layout()
plt.show()

## Step 10 — Conclusion

### Summary
- We built a simple 3-block CNN and trained it on the PneumoniaMNIST dataset to classify chest X-rays as **Normal** or **Pneumonia**.
- The model achieved reasonable accuracy after just 10 epochs of training, which we verified using accuracy/loss curves, a classification report, and a confusion matrix.
- We implemented **Grad-CAM from scratch** using `tf.GradientTape`, without relying on any external explainability library. The resulting heatmaps highlight the lung regions the model relies on for its predictions, giving us a way to sanity-check *why* the model made a given decision — not just *what* it decided.

### Why this matters
In medical imaging, a model's accuracy alone isn't enough — clinicians and researchers need to trust *how* a model reaches its conclusions. Grad-CAM provides a simple, visual way to audit the model's reasoning and catch cases where it might be focusing on irrelevant image regions instead of actual lung pathology.

### Possible extensions
- **Deeper / pretrained backbones**: swap the simple CNN for a pretrained model (e.g. ResNet, DenseNet via transfer learning) to potentially boost accuracy.
- **Data augmentation**: add rotations, flips, and zooms to make the model more robust and reduce overfitting.
- **Class imbalance handling**: use class weights or oversampling if the Normal/Pneumonia classes are imbalanced.
- **Higher-resolution images**: MedMNIST offers larger image sizes (64x64, 128x128, 224x224) for finer detail at the cost of more compute.
- **Other Grad-CAM variants**: try Grad-CAM++ or Score-CAM for potentially sharper, more localized heatmaps.
- **Multi-class extension**: apply the same pipeline to a multi-class chest disease dataset (e.g. ChestMNIST) instead of binary classification.